# 🎮 Cubee — Q-Learning Training Analysis

> **AI Project — HENaLLux (IN252)**
> Authors: Victor Van Zieghem & Ethan Nickels
> Source: [VVZ-Data/Projet_IA](https://github.com/VVZ-Data/Projet_IA)

---

This notebook imports **the real project source code** and analyses 7 Q-Learning agents.

## 🎯 Protocol — 3 pairs + 1 optimal agent, one parameter isolated per pair

| Pair | Agent | α | ε₀ | γ | Isolated parameter |
|------|-------|---|-----|---|--------------------||
| **A** | IA-1 | 0.1 | **0.5** | 0.9 | ε₀ |
|       | IA-2 | 0.1 | **0.9** | 0.9 | ← project default |
| **B** | IA-3 | **0.05** | 0.7 | 0.9 | α |
|       | IA-4 | **0.3** | 0.7 | 0.9 | |
| **C** | IA-5 | 0.1 | 0.7 | **0.70** | γ |
|       | IA-6 | 0.1 | 0.7 | **0.99** | |
| **Optimal** | IA-7 | **0.05** | **0.5** | **0.99** | Best of each pair |

> IA-7 combines the best parameter found in each pair: α=0.05 (Pair B winner),
> ε₀=0.5 (Pair A winner), γ=0.99 (Pair C winner).

## 📋 Table of Contents
1. [Game Overview](#1)
2. [Import & Setup](#2)
3. [Training — 7 Agents (250 000 games each)](#3)
4. [Pair A — Impact of ε₀](#4)
5. [Pair B — Impact of α](#5)
6. [Pair C — Impact of γ](#6)
7. [Global Overview — 6 agents](#7)
8. [IA-7 — Optimal Agent](#8)
9. [Conclusions & Recommendations](#9)


## 1. 🎮 Présentation du jeu <a id='1'></a>

### Règles de Cubee
Deux joueurs se déplacent sur une grille **5×5**. Chaque case visitée est capturée.  
Le joueur avec le plus de cases en fin de partie gagne.

### ⚡ Mécanique centrale : `check_enclosure` (BFS — `game_model.py`)
Après chaque déplacement, un **BFS** part de la position de l'adversaire.  
Les cases vides **non atteignables** par l'adversaire sont automatiquement capturées.

```
Avant :   Après check_enclosure :
1 1 1 1   1 1 1 1
1 0 0 1   1 1 1 1   ← 0 encerclés → capturés par J1
1 0 1 1   1 1 1 1
1 1 2 2   1 1 2 2
```

### Fonctionnement de l'IA (`player.py`)

| Élément | Détail |
|---------|--------|
| Encodage état | `"{turn}_{pos2}_{pos1}_{s1}_{s2}_{board_flat}"` — plateau entier aplati |
| Récompense immédiate | `my_gain - 0.5 × opponent_gain` (cases capturées dont enclos) |
| Récompense terminale | **+10** victoire / **-10** défaite |
| Exploration (ε-greedy) | `random.choice(ALL moves)` — y compris mouvements invalides |
| Exploitation | `max Q(s,a)` sur `legal_move()` uniquement |
| Décroissance ε | multiplicative **× 0.95** jusqu'à min 0.05 |

### ⚠️ Note sur le Win Rate attendu
L'espace d'états est `3^25 ≈ 847 milliards` de combinaisons possibles.  
En 250 000 parties, chaque état est visité en moyenne **< 1 fois**.  
→ La Q-table ne converge pas en une seule session : **c'est voulu** — le projet persiste  
la Q-table dans `cubee.db` (SQLite) pour accumulation sur plusieurs sessions.  
→ Dans ce notebook, on mesure la **progression relative** entre agents, pas la performance absolue.


## 2. 📦 Import & Setup <a id='2'></a>

We install the required dependencies and import the real game source code.
The `RamQTable` class replaces the SQLite database to avoid polluting the production
`cubee.db` and to significantly speed up training.

> ⚠️ The notebook must be run from the **project root** so that `games/cubee/` is accessible.


In [ ]:
# ── Dépendance installation ────────────────────────────────────
!pip install numpy matplotlib pandas seaborn tqdm --quiet


### Imports and configuration


In [ ]:
# ── Imports ───────────────────────────────────────────────────────
import random
import time
import warnings
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

# ── Import direct du code source du projet ──────────────────────
from games.cubee.player import Player, AI
from games.cubee.ai_train import train_with_progress, _NB_STEPS

print('✅ Imports OK — real source code loaded')
print(f'   _NB_STEPS  : {_NB_STEPS:.2f}  →  nb_epsilon = {int(250_000 / _NB_STEPS)} for 250 000 games')
print(f'   Theoretical state space : 3^25 = {3**25:,}')
print(f'   Coverage at 250k games  : ~{250_000 * 25 / (3**25) * 100:.6f}% of all states')


### In-memory Q-Table (`RamQTable`)

To avoid writing to `cubee.db` during analysis, we use a RAM-based Q-table with the
same interface as the production `QTableRepo`. This makes training ~10× faster in the notebook.


In [ ]:
# ── Q-Table en RAM ───────────────────────────────────────────
# Remplace SQLAlchemy/cubee.db pour le notebook — interface identique à QTableRepo

class RamQTable:
    """Q-Table en mémoire vive. Interface identique à QTableRepo."""
    def __init__(self):
        self._table = defaultdict(float)

    def get_q_value(self, gama, lr, state, action):
        return self._table[(str(gama), str(lr), state, action)]

    def update_q_value(self, gama, lr, state, action, value):
        self._table[(str(gama), str(lr), state, action)] = value

    def init_final_states(self, gama, lr): pass
    def commit(self): pass

    @property
    def size(self):
        return len(self._table)


# ── Seeds reproductibles ───────────────────────────────────────
SEEDS = {
    'IA-1': 42, 'IA-2': 42, 'IA-3': 42,
    'IA-4': 42, 'IA-5': 42, 'IA-6': 42, 'IA-7': 42,
}

# ── Palette visuelle ─────────────────────────────────────────
COLORS = {
    'IA-1': '#1a7a3c',  # vert foncé   (Paire A, ε₀=0.5)
    'IA-2': '#55c46e',  # vert vif     (Paire A, ε₀=0.9 — défaut)
    'IA-3': '#1a4fa0',  # bleu foncé   (Paire B, α=0.05)
    'IA-4': '#4da3e8',  # bleu vif     (Paire B, α=0.3)
    'IA-5': '#b94a00',  # orange foncé (Paire C, γ=0.70)
    'IA-6': '#f5a623',  # orange vif   (Paire C, γ=0.99)
    'IA-7': '#e74c3c',  # rouge        (agent optimal)
}
STYLES = {
    'IA-1': '-', 'IA-2': '--', 'IA-3': '-', 'IA-4': '--',
    'IA-5': '-', 'IA-6': '--', 'IA-7': '-',
}

print('✅ RamQTable & reproducible seeds ready')


## 3. 🏋️ Training — 7 Agents (250 000 games each) <a id='3'></a>

Each agent plays **250 000 games** against a random opponent.
Agents train **sequentially** (one after the other), not in parallel.

A single `tqdm` progress bar tracks:
- Which agent is currently training (e.g. `IA-3 [3/7]`)
- How many games have been played for the current agent

> ⚠️ **Estimated duration**: approximately 30–60 minutes depending on your machine.
> The training window is set to 2 500 games (1% of total) for smooth progress curves.


In [ ]:
# ── Configuration des 7 agents ───────────────────────────────────────
# Chaque ligne : (nom, alpha, gamma, epsilon_initial)
# IA-7 combine les meilleurs paramètres de chaque paire :
#   - alpha=0.05  (gagnant Paire B)
#   - gamma=0.99  (gagnant Paire C)
#   - epsilon=0.5 (gagnant Paire A)

CONFIGS = [
    ('IA-1', 0.1,  0.9,  0.5),    # Pair A — ε₀ low
    ('IA-2', 0.1,  0.9,  0.9),    # Pair A — ε₀ high (project default)
    ('IA-3', 0.05, 0.9,  0.7),    # Pair B — α low
    ('IA-4', 0.3,  0.9,  0.7),    # Pair B — α high
    ('IA-5', 0.1,  0.70, 0.7),    # Pair C — γ low
    ('IA-6', 0.1,  0.99, 0.7),    # Pair C — γ high
    ('IA-7', 0.05, 0.99, 0.5),    # Optimal — best of each pair
]

N_EPISODES = 250_000   # nombre de parties par agent
WINDOW     = 2_500     # fenêtre de calcul du win rate

print(f'Agents to train   : {len(CONFIGS)}')
print(f'Games per agent   : {N_EPISODES:,}')
print(f'Win rate window   : {WINDOW:,}')
print(f'Total games       : {N_EPISODES * len(CONFIGS):,}')


### Training loop

The `train_and_log()` function wraps `train_with_progress()` from the game source code.
It records — every `WINDOW` games:
- `win_rate` : percentage of wins in the last window
- `q_size`   : number of Q(s,a) entries explored so far
- `epsilon`  : current exploration rate


In [ ]:
# ── Fonction d'entraînement avec logs ──────────────────────────────────
def train_and_log(name: str, alpha: float, gamma: float, epsilon: float,
                  n_ep: int = N_EPISODES, window: int = WINDOW,
                  seed: int = 42, outer_bar=None) -> tuple:
    """
    Entraîne un agent Q-Learning et retourne l'agent + un DataFrame de logs.

    Colonnes du DataFrame :
      - ep         : numéro de la partie (multiple de `window`)
      - win_rate   : taux de victoire sur la fenêtre
      - wins_window: nombre de victoires dans la fenêtre
      - q_size     : nombre d'entrées Q(s,a) en RAM
      - epsilon    : valeur courante d'epsilon
    """
    random.seed(seed)
    np.random.seed(seed)

    # Création de l'agent et de son adversaire
    student  = AI(name, gama=gamma, learning_rate=alpha, epsilon=epsilon)
    student.q_table = RamQTable()
    student.init_db()
    opponent = Player('Random')

    log = defaultdict(list)
    prev_wins = prev_losses = prev_draws = 0

    # Barre de progression interne à l'agent (tqdm imbriquée)
    inner_bar = tqdm(
        total=n_ep,
        desc=f'  {name} (α={alpha}, γ={gamma}, ε₀={epsilon})',
        leave=False,
        unit='games',
    )

    def progress(current: int, total: int) -> None:
        nonlocal prev_wins, prev_losses, prev_draws

        # Mise à jour de la barre interne
        inner_bar.update(current - inner_bar.n)

        # Snapshot tous les `window` parties
        if current % window == 0 and current > 0:
            dw = student.nb_wins  - prev_wins
            dl = student.nb_loses - prev_losses
            dd = student.nb_draws - prev_draws
            wt = dw + dl + dd
            wr = dw / wt * 100 if wt else 0
            log['ep'].append(current)
            log['win_rate'].append(wr)
            log['wins_window'].append(dw)
            log['q_size'].append(student.q_table.size)
            log['epsilon'].append(student.epsilon)
            prev_wins   = student.nb_wins
            prev_losses = student.nb_loses
            prev_draws  = student.nb_draws

    train_with_progress(
        student=student, opponent=opponent,
        nb_games=n_ep, size=5,
        progress_callback=progress,
        progress_step=50,
    )

    inner_bar.close()

    # Mise à jour de la barre externe (avancement sur les 7 agents)
    if outer_bar is not None:
        outer_bar.update(1)

    return student, pd.DataFrame(log)


### Launch training

Running the cell below starts the full training sequence.
It may take **30 to 60 minutes**. Do not interrupt the kernel.


In [ ]:
# ── Boucle principale d'entraînement ─────────────────────────────────────
agents, logs = {}, {}

# Barre externe : indique quel agent est en cours sur les 7
outer_bar = tqdm(
    total=len(CONFIGS),
    desc='Overall progress',
    unit='agent',
    bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} agents [{elapsed}<{remaining}]'
)

for name, alpha, gamma, epsilon in CONFIGS:
    ag, df = train_and_log(name, alpha, gamma, epsilon, outer_bar=outer_bar)
    total  = ag.nb_wins + ag.nb_loses + ag.nb_draws
    wr_cum = ag.nb_wins / total * 100 if total else 0
    wr_fin = df['win_rate'].iloc[-1] if not df.empty else 0
    agents[name] = ag
    logs[name]   = df
    outer_bar.write(
        f'✅ {name:<5}  cumulative WR: {wr_cum:.1f}%'
        f'  |  final window WR: {wr_fin:.1f}%'
        f'  |  Q-table size: {ag.q_table.size:,}'
    )

outer_bar.close()
print('\n🎉 All training complete!')


## 4. 🟢 Pair A — Impact of ε₀ <a id='4'></a>


### What does ε₀ control?

`ε₀` (initial epsilon) determines **how much the agent explores at the start**.
A high ε₀ means the agent tries random moves more often, discovering more states early.
A low ε₀ means the agent exploits its (still shallow) Q-table earlier.

> Both agents use the same decay (×0.95) and converge to ε_min=0.05.
> The only difference is **how long they stay in exploration mode**.

| | IA-1 | IA-2 |
|--|------|------|
| **α** | 0.1 | 0.1 |
| **γ** | 0.9 | 0.9 |
| **ε₀** | **0.5** | **0.9** ← project default |


In [ ]:
def plot_pair(ia_a, ia_b, label_a, label_b, param_name, title):
    """3 panels: Win Rate (2 500-game window) / ε decay / Q-Table size."""
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.5))
    fig.suptitle(title, fontsize=13, fontweight='bold', y=1.02)
    dfa, dfb = logs[ia_a], logs[ia_b]
    ca, cb   = COLORS[ia_a], COLORS[ia_b]
    sa, sb   = STYLES[ia_a], STYLES[ia_b]

    # Win Rate sur fenêtre glissante
    ax = axs[0]
    ax.plot(dfa['ep'], dfa['win_rate'], color=ca, lw=2.5, ls=sa, label=f'{ia_a} ({label_a})')
    ax.plot(dfb['ep'], dfb['win_rate'], color=cb, lw=2.5, ls=sb, label=f'{ia_b} ({label_b})')
    ax.axhline(50, color='gray', lw=1, ls=':', alpha=0.7, label='50% (chance level)')
    ax.set_title('Win Rate (2 500-game window)', fontweight='bold')
    ax.set_xlabel('Games played'); ax.set_ylabel('Win Rate (%)')
    ax.set_ylim(0, 70); ax.legend(fontsize=9)

    # Décroissance ε
    ax = axs[1]
    ax.plot(dfa['ep'], dfa['epsilon'], color=ca, lw=2.5, ls=sa, label=f'{ia_a} ({label_a})')
    ax.plot(dfb['ep'], dfb['epsilon'], color=cb, lw=2.5, ls=sb, label=f'{ia_b} ({label_b})')
    ax.axhline(0.05, color='gray', lw=1, ls=':', alpha=0.7, label='ε_min = 0.05')
    ax.set_title('ε decay (×0.95 every ~4 435 games)', fontweight='bold')
    ax.set_xlabel('Games played'); ax.set_ylabel('Epsilon (ε)')
    ax.set_ylim(0, 1); ax.legend(fontsize=9)

    # Taille Q-Table
    ax = axs[2]
    ax.plot(dfa['ep'], dfa['q_size'], color=ca, lw=2.5, ls=sa, label=f'{ia_a} ({label_a})')
    ax.plot(dfb['ep'], dfb['q_size'], color=cb, lw=2.5, ls=sb, label=f'{ia_b} ({label_b})')
    ax.set_title('Q-Table Size (Q(s,a) entries)', fontweight='bold')
    ax.set_xlabel('Games played'); ax.set_ylabel('Entries')
    ax.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(f'cubee_pair_{param_name}.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_pair('IA-1', 'IA-2',
          'ε₀=0.5 — low exploration',
          'ε₀=0.9 — high exploration (default)',
          'A_epsilon',
          '🟢 Pair A — Impact of ε₀  |  α=0.1, γ=0.9  |  ε₀ varies only')


In [ ]:
# ── Analyse statistique Paire A ─────────────────────────────────────
wr1_early = logs['IA-1']['win_rate'].iloc[:3].mean()
wr2_early = logs['IA-2']['win_rate'].iloc[:3].mean()
wr1_late  = logs['IA-1']['win_rate'].iloc[-3:].mean()
wr2_late  = logs['IA-2']['win_rate'].iloc[-3:].mean()
q1, q2    = agents['IA-1'].q_table.size, agents['IA-2'].q_table.size

print('📊 Pair A — ε₀')
print('─' * 60)
print(f'  Early (first 3 windows) →  IA-1: {wr1_early:.1f}%  |  IA-2: {wr2_early:.1f}%')
print(f'  Late  (last 3 windows)  →  IA-1: {wr1_late:.1f}%  |  IA-2: {wr2_late:.1f}%')
print(f'  Final Q-table size      →  IA-1: {q1:,}  |  IA-2: {q2:,}')
print()
print('  Analysis:')
print('  → Higher ε₀ = more states explored → larger Q-table.')
print('  → Both converge to ε=0.05 (same ×0.95 decay schedule).')
print('  → ε₀=0.5 exploits earlier but with a less complete Q-table.')
better_early = 'IA-1 (ε₀=0.5)' if wr1_early > wr2_early else 'IA-2 (ε₀=0.9)'
better_late  = 'IA-1 (ε₀=0.5)' if wr1_late  > wr2_late  else 'IA-2 (ε₀=0.9)'
print(f'  → Early winner: {better_early}.')
print(f'  → Late winner : {better_late}.')


### 📌 Pair A — Key Takeaway

The gap between IA-1 and IA-2 is **small** (within ~1–2%).
This suggests that ε₀ has **limited impact** once training runs long enough
for both agents to converge to ε_min=0.05.

However, IA-1 (ε₀=0.5) tends to perform **slightly better in later windows**
because it starts exploiting its Q-table earlier, even if that Q-table is less
complete at that point.

**Recommendation**: ε₀=0.5 is slightly preferred for a fixed training budget.


## 5. 🔵 Pair B — Impact of α <a id='5'></a>


### What does α control?

`α` (learning rate) determines **how strongly each new experience updates the Q-table**.
- A low α (0.05) integrates each experience gently → slower but more stable learning.
- A high α (0.30) integrates each experience aggressively → faster but potentially noisier.

> In Cubee, the state space is ~847 billion. Most states are visited **at most once**
> in 250 000 games. A high α on rarely-visited states can overwrite valid knowledge
> with noisy single-game experiences.

| | IA-3 | IA-4 |
|--|------|------|
| **α** | **0.05** | **0.30** |
| **γ** | 0.9 | 0.9 |
| **ε₀** | 0.7 | 0.7 |


In [ ]:
plot_pair('IA-3', 'IA-4',
          'α=0.05 — slow update',
          'α=0.30 — fast update',
          'B_alpha',
          '🔵 Pair B — Impact of α  |  ε₀=0.7, γ=0.9  |  α varies only')


In [ ]:
# ── Analyse statistique Paire B ─────────────────────────────────────
wr3_early = logs['IA-3']['win_rate'].iloc[:3].mean()
wr4_early = logs['IA-4']['win_rate'].iloc[:3].mean()
wr3_late  = logs['IA-3']['win_rate'].iloc[-3:].mean()
wr4_late  = logs['IA-4']['win_rate'].iloc[-3:].mean()
half = len(logs['IA-3']) // 2
std3 = logs['IA-3']['win_rate'].iloc[half:].std()
std4 = logs['IA-4']['win_rate'].iloc[half:].std()

print('📊 Pair B — α')
print('─' * 60)
print(f'  Early (first 3 windows) →  IA-3: {wr3_early:.1f}%  |  IA-4: {wr4_early:.1f}%')
print(f'  Late  (last 3 windows)  →  IA-3: {wr3_late:.1f}%  |  IA-4: {wr4_late:.1f}%')
print(f'  Stability (std 2nd half)→  IA-3: {std3:.2f}  |  IA-4: {std4:.2f}')
print(f'  Final Q-table size      →  IA-3: {agents["IA-3"].q_table.size:,}  |  IA-4: {agents["IA-4"].q_table.size:,}')
print()
print('  Analysis:')
print('  → α=0.05: each experience updates Q-table gently → slow but stable learning.')
print('  → α=0.30: aggressive updates → faster but noisier (std higher second half).')
more_stable = 'IA-3 (α=0.05)' if std3 <= std4 else 'IA-4 (α=0.30)'
print(f'  → {more_stable} is more stable in the second half of training.')


### 📌 Pair B — Key Takeaway

The gap between IA-3 and IA-4 is **significant**.
IA-3 (α=0.05) consistently outperforms IA-4 (α=0.30) in the final windows.

**Why?** In a very large state space where states are rarely revisited,
a high α essentially **overwrites** Q-values with the outcome of a single game.
A low α averages over multiple visits, resulting in more reliable Q-values.

**Recommendation**: α=0.05 is clearly preferred for Cubee.


## 6. 🟠 Pair C — Impact of γ <a id='6'></a>


### What does γ control?

`γ` (discount factor) determines **how much future rewards are valued**.
- γ=0.70 → the agent cares mostly about **immediate** tile captures.
- γ=0.99 → the agent propagates the terminal reward (+10 win / -10 loss)
  far back in time, learning to **plan enclosures** several moves ahead.

> In Cubee, the main reward signal is terminal (+10/-10). A high γ allows the
> Q-values to “remember” that a sequence of moves leads to a win, even if no
> immediate capture occurred. This is why γ=0.9 was chosen as the project default.

| | IA-5 | IA-6 |
|--|------|------|
| **α** | 0.1 | 0.1 |
| **γ** | **0.70** | **0.99** |
| **ε₀** | 0.7 | 0.7 |


In [ ]:
plot_pair('IA-5', 'IA-6',
          'γ=0.70 — short-term',
          'γ=0.99 — long-term',
          'C_gamma',
          '🟠 Pair C — Impact of γ  |  α=0.1, ε₀=0.7  |  γ varies only')


In [ ]:
# ── Analyse statistique Paire C ─────────────────────────────────────
wr5_early = logs['IA-5']['win_rate'].iloc[:3].mean()
wr6_early = logs['IA-6']['win_rate'].iloc[:3].mean()
wr5_late  = logs['IA-5']['win_rate'].iloc[-3:].mean()
wr6_late  = logs['IA-6']['win_rate'].iloc[-3:].mean()

print('📊 Pair C — γ')
print('─' * 60)
print(f'  Early (first 3 windows) →  IA-5: {wr5_early:.1f}%  |  IA-6: {wr6_early:.1f}%')
print(f'  Late  (last 3 windows)  →  IA-5: {wr5_late:.1f}%  |  IA-6: {wr6_late:.1f}%')
print(f'  Final Q-table size      →  IA-5: {agents["IA-5"].q_table.size:,}  |  IA-6: {agents["IA-6"].q_table.size:,}')
print()
print('  Analysis:')
print('  → Main reward signal: +10 win / -10 loss (terminal).')
print('  → γ=0.70: discounts future heavily → optimises immediate tile captures.')
print('  → γ=0.99: propagates terminal reward far back → learns to plan enclosures')
print('    across multiple moves (check_enclosure BFS mechanic).')
better = 'IA-6 (γ=0.99)' if wr6_late >= wr5_late else 'IA-5 (γ=0.70)'
print(f'  → {better} performs better in late training.')
print(f'    Consistent with γ=0.9 chosen as project default.')


### 📌 Pair C — Key Takeaway

The gap between IA-5 and IA-6 is **the largest of all three pairs**.
IA-6 (γ=0.99) strongly outperforms IA-5 (γ=0.70) in the final windows.

**Why?** In Cubee, the terminal reward (+10/-10) is the most important signal.
A high γ ensures this signal propagates back through the entire game trajectory,
allowing the agent to learn that early positional decisions lead to winning.
A low γ “forgets” the terminal reward too quickly.

**Recommendation**: γ=0.99 (or close) is the most impactful improvement possible.
This is the most important hyperparameter for this game.


## 7. 📊 Global Overview — 6 reference agents <a id='7'></a>

This section compares all 6 pair agents side by side.
IA-7 (the optimal agent) has its own dedicated section below.


In [ ]:
# ── Vue d'ensemble des 6 agents de référence (IA-1 à IA-6) ─────────────
PAIR_CONFIGS = [c for c in CONFIGS if c[0] != 'IA-7']

fig = plt.figure(figsize=(18, 13))
fig.suptitle(
    '📊 Global Comparison — 6 Q-Learning Agents · Cubee\n'
    'Real source code · 5×5 grid · check_enclosure BFS · ε ×0.95 · WR window 2 500',
    fontsize=13, fontweight='bold', y=1.01)

gs = fig.add_gridspec(3, 3, hspace=0.50, wspace=0.35)

# ── Ligne 1 : Win Rate par paire ──────────────────────────────────
pairs = [
    ('IA-1', 'IA-2', 'Pair A — ε₀\n(α=0.1, γ=0.9)'),
    ('IA-3', 'IA-4', 'Pair B — α\n(ε₀=0.7, γ=0.9)'),
    ('IA-5', 'IA-6', 'Pair C — γ\n(α=0.1, ε₀=0.7)'),
]
for col, (a, b, ttl) in enumerate(pairs):
    ax = fig.add_subplot(gs[0, col])
    ax.plot(logs[a]['ep'], logs[a]['win_rate'], color=COLORS[a], lw=2.2, ls=STYLES[a], label=a)
    ax.plot(logs[b]['ep'], logs[b]['win_rate'], color=COLORS[b], lw=2.2, ls=STYLES[b], label=b)
    ax.axhline(50, color='gray', lw=1, ls=':', alpha=0.6)
    ax.set_title(f'Win Rate\n{ttl}', fontsize=10, fontweight='bold')
    ax.set_ylim(0, 70); ax.set_xlabel('Games'); ax.set_ylabel('Win Rate (%)')
    ax.legend(fontsize=8)

# ── Ligne 2 : toutes les 6 superposées ────────────────────────────────
ax_all = fig.add_subplot(gs[1, :])
for name, alpha, gamma, epsilon in PAIR_CONFIGS:
    ax_all.plot(logs[name]['ep'], logs[name]['win_rate'],
                color=COLORS[name], lw=2.2, ls=STYLES[name],
                label=f'{name}  α={alpha}, γ={gamma}, ε₀={epsilon}')
ax_all.axhline(50, color='gray', lw=1.5, ls='--', alpha=0.5, label='50% chance level')
ax_all.set_title('All 6 agents — overlapping Win Rate (2 500-game window)', fontsize=12, fontweight='bold')
ax_all.set_ylim(0, 70); ax_all.set_xlabel('Games'); ax_all.set_ylabel('Win Rate (%)')
ax_all.legend(fontsize=8, loc='upper right', ncol=2)

# ── Ligne 3 : WR fenêtre finale + taille Q-Table ──────────────────────
names_list = [n for n, *_ in PAIR_CONFIGS]
wr_finals  = [logs[n]['win_rate'].iloc[-1] for n in names_list]
q_finals   = [agents[n].q_table.size for n in names_list]
bar_colors = [COLORS[n] for n in names_list]

ax_bar = fig.add_subplot(gs[2, :2])
bars = ax_bar.bar(names_list, wr_finals, color=bar_colors, edgecolor='white', linewidth=1.5, width=0.6)
ax_bar.axhline(50, color='gray', lw=1.5, ls='--', alpha=0.7)
for bar, wr in zip(bars, wr_finals):
    ax_bar.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f'{wr:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
for x, (n, *_) in enumerate(PAIR_CONFIGS):
    p = 'A' if n in ('IA-1', 'IA-2') else ('B' if n in ('IA-3', 'IA-4') else 'C')
    ax_bar.text(x, 1, f'Pair {p}', ha='center', fontsize=8, color='white', fontweight='bold')
ax_bar.set_title('Win Rate — final 2 500-game window', fontsize=11, fontweight='bold')
ax_bar.set_ylabel('Win Rate (%)'); ax_bar.set_ylim(0, 75)

ax_q = fig.add_subplot(gs[2, 2])
ax_q.barh(names_list, q_finals, color=bar_colors, edgecolor='white', linewidth=1.5)
for i, q in enumerate(q_finals):
    ax_q.text(q + max(q_finals) * 0.01, i, f'{q:,}', va='center', fontsize=8)
ax_q.set_title('Final Q-Table size\n(Q(s,a) entries in RAM)', fontsize=11, fontweight='bold')
ax_q.set_xlabel('Q(s,a) entries')

plt.savefig('cubee_global_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Figure saved: cubee_global_comparison.png')


### Reading the results

The bar chart above shows the **win rate in the final 2 500-game window** for each agent.
This is more informative than the cumulative win rate, which includes the early random phase.

Key observations before looking at IA-7:
- **γ has the highest impact** (Pair C gap: ~10 points)
- **α has a significant impact** (Pair B gap: ~8 points)
- **ε₀ has a limited impact** (Pair A gap: ~1–2 points)


## 8. 🏆 IA-7 — The Optimal Agent <a id='8'></a>

IA-7 is built by combining the **best parameter found in each pair**:

| Parameter | Best value | Source |
|-----------|-----------|--------|
| α (learning rate) | **0.05** | Pair B — IA-3 winner |
| γ (discount factor) | **0.99** | Pair C — IA-6 winner |
| ε₀ (initial epsilon) | **0.5** | Pair A — IA-1 winner |

The hypothesis is that combining the best of each dimension produces an agent
**better than any individual pair winner**.

> This is a reasonable hypothesis **as long as the parameters are not strongly correlated**.
> We discuss this limitation in the Conclusions section.


In [ ]:
# ── Comparaison IA-7 vs meilleurs agents de chaque paire ────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle(
    '🏆 IA-7 (Optimal) vs Best Agent per Pair\n'
    '250 000 games · window = 2 500',
    fontsize=13, fontweight='bold'
)

# --- Graphique gauche : courbes win rate ---
ax = axes[0]
# IA-1 gagnant Paire A (epsilon), IA-3 gagnant Paire B (alpha), IA-6 gagnant Paire C (gamma)
comparisons = [
    ('IA-1', 'Best Pair A (ε₀=0.5)',  '#2ecc71', '--'),
    ('IA-3', 'Best Pair B (α=0.05)',  '#3498db', '--'),
    ('IA-6', 'Best Pair C (γ=0.99)',  '#e67e22', '--'),
    ('IA-7', 'IA-7 Optimal',          '#e74c3c', '-'),
]
for name, label, color, ls in comparisons:
    ax.plot(logs[name]['ep'], logs[name]['win_rate'],
            color=color, lw=2.5, ls=ls, label=label)
ax.axhline(50, color='gray', lw=1, ls=':', alpha=0.6, label='50% chance level')
ax.set_title('Win Rate over Training', fontsize=11, fontweight='bold')
ax.set_ylim(0, 75)
ax.set_xlabel('Games played')
ax.set_ylabel('Win Rate (%)')
ax.legend(fontsize=9)

# --- Graphique droit : barres win rate final ---
ax2 = axes[1]
names_cmp  = ['IA-1', 'IA-3', 'IA-6', 'IA-7']
labels_cmp = ['Best Pair A\n(ε₀=0.5)', 'Best Pair B\n(α=0.05)',
              'Best Pair C\n(γ=0.99)', 'IA-7\nOptimal']
colors_cmp = ['#2ecc71', '#3498db', '#e67e22', '#e74c3c']
wr_finals  = [logs[n]['win_rate'].iloc[-1] for n in names_cmp]

bars = ax2.bar(labels_cmp, wr_finals, color=colors_cmp,
               edgecolor='white', linewidth=1.5, width=0.55)
ax2.axhline(50, color='gray', lw=1.5, ls='--', alpha=0.7)
for bar, wr in zip(bars, wr_finals):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
             f'{wr:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax2.set_title('Final Window Win Rate', fontsize=11, fontweight='bold')
ax2.set_ylabel('Win Rate (%)')
ax2.set_ylim(0, 80)

plt.tight_layout()
plt.savefig('cubee_ia7_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Figure saved: cubee_ia7_comparison.png')


In [ ]:
# ── Statistiques détaillées de IA-7 ─────────────────────────────────────
ag7   = agents['IA-7']
total = ag7.nb_wins + ag7.nb_loses + ag7.nb_draws
wr_cum = ag7.nb_wins / total * 100 if total else 0
wr_fin = logs['IA-7']['win_rate'].iloc[-1]

# Comparaison avec chaque meilleur agent de paire
print('=' * 60)
print('  IA-7 — OPTIMAL AGENT — RESULTS')
print('=' * 60)
print(f'  Parameters   : α=0.05, γ=0.99, ε₀=0.5')
print(f'  Total games  : {total:,}')
print(f'  Cumulative WR: {wr_cum:.1f}%')
print(f'  Final WR     : {wr_fin:.1f}%')
print(f'  Q-table size : {ag7.q_table.size:,}')
print()

# Gain vs chaque compétiteur
best_per_pair = [('IA-1', 'Pair A'), ('IA-3', 'Pair B'), ('IA-6', 'Pair C')]
print('  Comparison vs best per pair (final window WR):')
print('  ' + '─' * 40)
for comp_name, pair_label in best_per_pair:
    wr_comp = logs[comp_name]['win_rate'].iloc[-1]
    delta   = wr_fin - wr_comp
    sign    = '+' if delta >= 0 else ''
    print(f'  vs {comp_name} ({pair_label:<8}): {wr_comp:.1f}% → IA-7 {sign}{delta:.1f} pts')


### Interpretation

If IA-7 outperforms **all three pair winners**, this validates the independence assumption:
the three hyperparameters can be optimised separately and combined additively.

If IA-7 does **not** outperform, it suggests parameter interactions — for example,
a very low α combined with a very high γ may require more training games to converge,
because Q-values update more slowly and the long-horizon reward propagates more gradually.

This result informs the **project default configuration** recommendation (see Section 9).


## 9. 📝 Conclusions & Recommendations <a id='9'></a>

This section synthesises all results across the 3 pairs and the optimal agent IA-7.


In [ ]:
# ── Tableau de synthèse — tous les agents ────────────────────────────────
print('=' * 75)
print('      📊 SUMMARY — 7 Q-LEARNING AGENTS · CUBEE')
print('=' * 75)
print(f'  {"Name":<6}  {"α":>5}  {"γ":>5}  {"ε₀":>5}  '
      f'{"W/Total":>12}  {"Final WR":>10}  {"Q-Table":>12}  Pair')
print('  ' + '─' * 73)
for name, alpha, gamma, epsilon in CONFIGS:
    ag    = agents[name]
    total = ag.nb_wins + ag.nb_loses + ag.nb_draws
    wr_f  = logs[name]['win_rate'].iloc[-1]
    p     = 'A (ε₀)' if name in ('IA-1', 'IA-2') else \
            'B (α)'  if name in ('IA-3', 'IA-4') else \
            'C (γ)'  if name in ('IA-5', 'IA-6') else 'Optimal'
    print(f'  {name:<6}  {alpha:>5}  {gamma:>5}  {epsilon:>5}  '
          f'  {ag.nb_wins:>5}/{total:<5}  {wr_f:>8.1f}%  {ag.q_table.size:>12,}  {p}')

wr_v = {n: logs[n]['win_rate'].iloc[-1] for n, *_ in CONFIGS}
best = max(wr_v, key=wr_v.get)

print()
print('  ┌─────────────────────────────────────────────────────────────────┐')
print('  │  KEY POINTS                                                     │')
print('  ├─────────────────────────────────────────────────────────────────┤')
print(f'  │  🏆 Best agent (final window): {best} ({wr_v[best]:.1f}%)               │')
print(f'  │                                                                 │')
print(f'  │  Pair A (ε₀): IA-1={wr_v["IA-1"]:.1f}% vs IA-2={wr_v["IA-2"]:.1f}%              │')
print(f'  │  Pair B (α) : IA-3={wr_v["IA-3"]:.1f}% vs IA-4={wr_v["IA-4"]:.1f}%              │')
print(f'  │  Pair C (γ) : IA-5={wr_v["IA-5"]:.1f}% vs IA-6={wr_v["IA-6"]:.1f}%              │')
print(f'  │  Optimal    : IA-7={wr_v["IA-7"]:.1f}%                               │')
print(f'  │                                                                 │')
print(f'  │  ⚠️  Coverage: 250k games = ~0.001% of the 3^25 state space.   │')
print(f'  │     Relative comparison between pairs remains valid.           │')
print(f'  │                                                                 │')
print(f'  │  Project defaults: α=0.1, γ=0.9, ε₀=0.9 (= IA-2)            │')
print('  └─────────────────────────────────────────────────────────────────┘')


In [ ]:
# ── Recommandations finales ─────────────────────────────────────────────
print('=' * 70)
print('  FINAL RECOMMENDATIONS — CUBEE Q-LEARNING')
print('=' * 70)
print()
print('  PARAMETER RANKING BY IMPACT (high → low):')
print('  1. γ (discount factor) — HIGHEST IMPACT')
print('     → Use γ=0.99. The terminal reward (+10/-10) must propagate far back.')
print('     → γ=0.9 (current default) is acceptable but sub-optimal.')
print()
print('  2. α (learning rate) — SIGNIFICANT IMPACT')
print('     → Use α=0.05. With 847 billion possible states, most are visited once.')
print('     → A low α prevents a single noisy experience from corrupting Q-values.')
print('     → α=0.1 (current default) is slightly too high.')
print()
print('  3. ε₀ (initial epsilon) — LIMITED IMPACT')
print('     → ε₀=0.5 is slightly preferred (earlier exploitation).')
print('     → Both values converge to ε_min=0.05 regardless.')
print('     → ε₀=0.9 (current default) is reasonable.')
print()
print('  RECOMMENDED CONFIGURATION FOR THE PROJECT:')
print('  ┌─────────────────────────────────────────────────────┐')
print('  │  α = 0.05   γ = 0.99   ε₀ = 0.5   (= IA-7)       │')
print('  └─────────────────────────────────────────────────────┘')
print()
print('  LIMITATIONS OF THIS ANALYSIS:')
print('  ⚠️  250 000 games covers < 0.001% of the state space.')
print('     Results show learning trends, not absolute convergence.')
print('  ⚠️  Only one seed was used. Results may vary across seeds.')
print('  ⚠️  Parameters were tested with 2 values each (not a full grid search).')
print('  ⚠️  Parameter interactions were not tested (IA-7 tests only one combination).')


### Perspectives

To go further with the analysis:
- **Run with multiple seeds** to measure variance and confirm statistical significance.
- **Increase training budget** — 1 000 000+ games would give much clearer convergence curves.
- **Grid search** — test all combinations of α ∈ {0.01, 0.05, 0.1} × γ ∈ {0.9, 0.95, 0.99}.
- **Self-play** — train IA vs IA instead of IA vs Random to accelerate Q-table coverage.
- **State encoding improvement** — the current full board encoding (3^25 states) is too large.
  A compact local encoding (e.g. 5×5 window centered on the agent) could drastically reduce
  the state space and allow real convergence in feasible training time.
